# 07 — Monte Carlo Season Simulation

This notebook simulates the 2026 NFL regular season using the game level win probabilities created in Notebook 06.

Each simulation plays all 272 scheduled games once. The outcome of each game is sampled according to the model's predicted win probability, allowing favorites to lose and underdogs to win.

Repeating the season thousands of times produces a distribution of possible records for every team rather than a single deterministic forecast.

In [92]:
from pathlib import Path

import numpy as np
import pandas as pd

In [93]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

In [94]:
game_predictions = pd.read_parquet(
    PROCESSED_DIR / "2026_game_predictions.parquet"
)

In [95]:
teams = sorted(
    set(game_predictions["home_team"])
    | set(game_predictions["away_team"])
)

N_SIMULATIONS = 10000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

## Strength of Schedule

Schedule difficulty is already reflected in each game's predicted win probability because every matchup uses the projected strength of both teams.

For interpretation, each team's preseason strength of schedule is also calculated as the average projected strength of its 17 opponents. Higher values indicate a more difficult schedule.

In [96]:
sos_rows = []

for team in teams:
    team_games = game_predictions[
        (game_predictions["home_team"] == team)
        | (game_predictions["away_team"] == team)
    ].copy()

    opponent_strengths = np.where(
        team_games["home_team"] == team,
        team_games["away_team_strength"],
        team_games["home_team_strength"]
    )

    sos_rows.append({
        "team": team,
        "strength_of_schedule": opponent_strengths.mean()
    })

strength_of_schedule = pd.DataFrame(sos_rows)

## Simulate Regular Seasons

For each simulated season, every game's outcome is generated using its predicted home team win probability.

A random value below the home win probability produces a home win; otherwise the away team wins. Team win totals are recorded after all 272 games have been simulated.

In [97]:
team_index = {
    team: i
    for i, team in enumerate(teams)
}

simulated_wins = np.zeros(
    (N_SIMULATIONS, len(teams)),
    dtype=np.int16
)

home_teams = game_predictions["home_team"].to_numpy()
away_teams = game_predictions["away_team"].to_numpy()
home_probs = game_predictions["home_win_probability"].to_numpy()

for sim in range(N_SIMULATIONS):
    home_wins = rng.random(len(game_predictions)) < home_probs

    for game_idx, home_win in enumerate(home_wins):
        winner = (
            home_teams[game_idx]
            if home_win
            else away_teams[game_idx]
        )

        simulated_wins[
            sim,
            team_index[winner]
        ] += 1

## Projected Team Records

Each team's projected win total is the average number of wins across all simulated seasons.

The simulation distribution also provides lower and upper ranges that show how much uncertainty exists around each team's preseason expectation.

In [98]:
team_projection_rows = []

for team, idx in team_index.items():
    wins = simulated_wins[:, idx]

    team_projection_rows.append({
        "team": team,
        "projected_wins": wins.mean(),
        "median_wins": np.median(wins),
        "win_p10": np.percentile(wins, 10),
        "win_p90": np.percentile(wins, 90)
    })

season_projections = (
    pd.DataFrame(team_projection_rows)
    .sort_values(
        "projected_wins",
        ascending=False
    )
    .reset_index(drop=True)
)

season_projections["projected_losses"] = (
    17 - season_projections["projected_wins"]
)

season_projections["win_rank"] = (
    season_projections.index + 1
)

In [99]:
season_projections = (
    season_projections
    .merge(
        strength_of_schedule,
        on="team",
        how="left"
    )
)

In [100]:
season_projections["strength_of_schedule_rank"] = (
    season_projections["strength_of_schedule"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

## Representative Final Standings

The Monte Carlo simulation produces thousands of possible NFL seasons. The model's average win totals are useful for measuring expected team strength, but the final project also requires one exact predicted record for every team.

To select a single representative season, simulated seasons are first filtered to those with league wide record dispersion similar to recent NFL seasons. Among those realistic seasons, the model selects the simulation whose team records are closest overall to the model's expected win totals.

This preserves the model's team rankings and game probabilities while producing one internally consistent set of final standings.

In [101]:
historical_standings_sd = np.array([
    3.678,
    2.882,
    3.125,
    2.747,
    3.707,
    3.462
])

HISTORICAL_STANDINGS_SD_MEAN = (
    historical_standings_sd.mean()
)

HISTORICAL_STANDINGS_SD_MIN = (
    historical_standings_sd.min()
)

HISTORICAL_STANDINGS_SD_MAX = (
    historical_standings_sd.max()
)

print(
    "Historical standings SD mean:",
    round(
        HISTORICAL_STANDINGS_SD_MEAN,
        3
    )
)

print(
    "Historical standings SD range:",
    round(
        HISTORICAL_STANDINGS_SD_MIN,
        3
    ),
    "to",
    round(
        HISTORICAL_STANDINGS_SD_MAX,
        3
    )
)

Historical standings SD mean: 3.267
Historical standings SD range: 2.747 to 3.707


In [102]:
projected_wins_by_team = np.array([
    season_projections
    .set_index("team")
    .loc[teams, "projected_wins"]
])

simulation_standings_sd = (
    simulated_wins.std(
        axis=1,
        ddof=1
    )
)

realistic_season_mask = (
    (
        simulation_standings_sd
        >= HISTORICAL_STANDINGS_SD_MIN
    )
    &
    (
        simulation_standings_sd
        <= HISTORICAL_STANDINGS_SD_MAX
    )
)

realistic_simulation_indices = np.where(
    realistic_season_mask
)[0]

print(
    "Realistic simulations:",
    len(
        realistic_simulation_indices
    ),
    "of",
    N_SIMULATIONS
)

Realistic simulations: 2103 of 10000


In [103]:
realistic_simulated_wins = (
    simulated_wins[
        realistic_simulation_indices
    ]
)

squared_distance_from_projection = (
    (
        realistic_simulated_wins
        - projected_wins_by_team
    ) ** 2
).sum(
    axis=1
)

best_realistic_position = (
    np.argmin(
        squared_distance_from_projection
    )
)

representative_simulation_index = (
    realistic_simulation_indices[
        best_realistic_position
    ]
)

representative_wins = (
    simulated_wins[
        representative_simulation_index
    ]
)

print(
    "Representative simulation:",
    representative_simulation_index
)

print(
    "Representative standings SD:",
    round(
        representative_wins.std(
            ddof=1
        ),
        3
    )
)

print(
    "Best record:",
    representative_wins.max()
)

print(
    "Worst record:",
    representative_wins.min()
)

Representative simulation: 6717
Representative standings SD: 2.782
Best record: 15
Worst record: 3


In [104]:
representative_standings = pd.DataFrame({
    "team": teams,
    "predicted_wins": representative_wins
})

representative_standings[
    "predicted_losses"
] = (
    17
    - representative_standings[
        "predicted_wins"
    ]
)

representative_standings = (
    representative_standings
    .merge(
        season_projections[
            [
                "team",
                "projected_wins"
            ]
        ],
        on="team",
        how="left"
    )
    .sort_values(
        [
            "predicted_wins",
            "projected_wins"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

representative_standings[
    "prediction_rank"
] = (
    representative_standings.index
    + 1
)

representative_standings = (
    representative_standings[
        [
            "prediction_rank",
            "team",
            "predicted_wins",
            "predicted_losses",
            "projected_wins"
        ]
    ]
)

representative_standings

,prediction_rank,team,predicted_wins,predicted_losses,projected_wins
0,1,BUF,15,2,10.5924
1,2,PHI,13,4,9.8576
2,3,DET,12,5,10.7638
3,4,LA,12,5,10.3205
4,5,SEA,11,6,10.5548
5,6,HOU,11,6,9.9525
6,7,SF,11,6,9.5526
7,8,BAL,10,7,10.2017
8,9,DEN,10,7,10.1649
9,10,IND,10,7,8.9828


In [105]:
print(
    "Total predicted wins:",
    representative_standings[
        "predicted_wins"
    ].sum()
)

print(
    "Total predicted losses:",
    representative_standings[
        "predicted_losses"
    ].sum()
)

print(
    "Standings SD:",
    round(
        representative_standings[
            "predicted_wins"
        ].std(),
        3
    )
)

print(
    "Best predicted record:",
    int(
        representative_standings[
            "predicted_wins"
        ].max()
    ),
    "-",
    int(
        representative_standings.loc[
            representative_standings[
                "predicted_wins"
            ].idxmax(),
            "predicted_losses"
        ]
    )
)

print(
    "Worst predicted record:",
    int(
        representative_standings[
            "predicted_wins"
        ].min()
    ),
    "-",
    int(
        representative_standings.loc[
            representative_standings[
                "predicted_wins"
            ].idxmin(),
            "predicted_losses"
        ]
    )
)

Total predicted wins: 272
Total predicted losses: 272
Standings SD: 2.782
Best predicted record: 15 - 2
Worst predicted record: 3 - 14


### Final Standings Prediction

The final standings represent one complete simulated NFL season rather than the average result across all simulations.

The selected season is restricted to a historically realistic level of league wide win dispersion and is then chosen as the simulation closest overall to the model's expected team win totals. This produces exact, internally consistent records while preserving the underlying game probabilities and team-strength estimates.

Expected wins remain useful for measuring underlying team quality, while predicted wins and losses represent the model's official 2026 standings forecast.

## Playoff Simulation

Each simulated season is converted into a playoff field using the NFL's current seven team format.

The four division winners in each conference qualify automatically, followed by the three best remaining teams as wild cards. Because full NFL tiebreaking procedures require additional game-level criteria, ties in record are resolved randomly within the simulation.

This provides an approximation of playoff qualification while keeping the simulation understandable and reproducible.

In [106]:
divisions = {
    "AFC East": ["BUF", "MIA", "NE", "NYJ"],
    "AFC North": ["BAL", "CIN", "CLE", "PIT"],
    "AFC South": ["HOU", "IND", "JAX", "TEN"],
    "AFC West": ["DEN", "KC", "LAC", "LV"],
    "NFC East": ["DAL", "NYG", "PHI", "WAS"],
    "NFC North": ["CHI", "DET", "GB", "MIN"],
    "NFC South": ["ATL", "CAR", "NO", "TB"],
    "NFC West": ["ARI", "LA", "SEA", "SF"]
}

conferences = {
    "AFC": [
        "BUF", "MIA", "NE", "NYJ",
        "BAL", "CIN", "CLE", "PIT",
        "HOU", "IND", "JAX", "TEN",
        "DEN", "KC", "LAC", "LV"
    ],
    "NFC": [
        "DAL", "NYG", "PHI", "WAS",
        "CHI", "DET", "GB", "MIN",
        "ATL", "CAR", "NO", "TB",
        "ARI", "LA", "SEA", "SF"
    ]
}

In [107]:
playoff_counts = {
    team: 0
    for team in teams
}

division_counts = {
    team: 0
    for team in teams
}

In [108]:
for sim in range(N_SIMULATIONS):
    sim_wins = {
        team: simulated_wins[
            sim,
            team_index[team]
        ]
        for team in teams
    }

    division_winners = []

    for division_teams in divisions.values():
        shuffled = rng.permutation(division_teams)

        winner = max(
            shuffled,
            key=lambda team: sim_wins[team]
        )

        division_winners.append(winner)
        division_counts[winner] += 1

    for conference, conference_teams in conferences.items():
        conference_division_winners = [
            team
            for team in division_winners
            if team in conference_teams
        ]

        remaining_teams = [
            team
            for team in conference_teams
            if team not in conference_division_winners
        ]

        shuffled_remaining = rng.permutation(
            remaining_teams
        )

        wild_cards = sorted(
            shuffled_remaining,
            key=lambda team: sim_wins[team],
            reverse=True
        )[:3]

        playoff_teams = (
            conference_division_winners
            + wild_cards
        )

        for team in playoff_teams:
            playoff_counts[team] += 1

In [109]:
season_projections["division_win_probability"] = (
    season_projections["team"]
    .map(division_counts)
    / N_SIMULATIONS
)

season_projections["playoff_probability"] = (
    season_projections["team"]
    .map(playoff_counts)
    / N_SIMULATIONS
)

In [110]:
playoff_review = season_projections[
    [
        "team",
        "projected_wins",
        "division_win_probability",
        "playoff_probability"
    ]
].copy()

print(
    playoff_review
    .sort_values(
        "playoff_probability",
        ascending=False
    )
    .round(3)
    .to_string(index=False)
)

print()
print(
    "Total expected playoff teams:",
    round(
        season_projections[
            "playoff_probability"
        ].sum(),
        3
    )
)

print(
    "Total expected division winners:",
    round(
        season_projections[
            "division_win_probability"
        ].sum(),
        3
    )
)

team  projected_wins  division_win_probability  playoff_probability
 BUF          10.592                     0.632                0.816
 DET          10.764                     0.510                0.790
 SEA          10.555                     0.412                0.760
 BAL          10.202                     0.519                0.745
 DEN          10.165                     0.533                0.743
  LA          10.320                     0.354                0.718
 HOU           9.952                     0.444                0.702
 PHI           9.858                     0.565                0.688
 JAX           9.328                     0.298                0.587
  SF           9.553                     0.216                0.579
  NE           9.222                     0.288                0.570
  GB           9.498                     0.246                0.564
  TB           8.931                     0.419                0.521
  KC           8.942                     0.257  

## Final Season Projections

The final preseason projections summarize each team's results across 10,000 simulated seasons.

Projected wins represent the average simulated record, while the 10th and 90th percentiles show a range of plausible outcomes. Strength of schedule measures the average projected strength of each team's opponents, while division and playoff probabilities represent how often each outcome occurred across the simulations.

In [111]:
season_projections_2026 = (
    season_projections[
        [
            "team",
            "projected_wins",
            "projected_losses",
            "median_wins",
            "win_p10",
            "win_p90",
            "strength_of_schedule",
            "strength_of_schedule_rank",
            "division_win_probability",
            "playoff_probability"
        ]
    ]
    .sort_values(
        "projected_wins",
        ascending=False
    )
    .reset_index(drop=True)
)

season_projections_2026["projection_rank"] = (
    season_projections_2026.index + 1
)

season_projections_2026 = season_projections_2026[
    [
        "projection_rank",
        "team",
        "projected_wins",
        "projected_losses",
        "median_wins",
        "win_p10",
        "win_p90",
        "strength_of_schedule",
        "strength_of_schedule_rank",
        "division_win_probability",
        "playoff_probability"
    ]
]

In [112]:
season_projections_2026.to_parquet(
    PROCESSED_DIR / "2026_season_projections.parquet",
    index=False
)
representative_standings.to_parquet(
    PROCESSED_DIR
    / "2026_predicted_standings.parquet",
    index=False
)

In [113]:
simulated_wins_2026 = pd.DataFrame(
    simulated_wins,
    columns=teams
)

simulated_wins_2026.to_parquet(
    PROCESSED_DIR / "2026_simulated_wins.parquet",
    index=False
)

In [114]:
print(
    "Teams:",
    len(season_projections_2026)
)

print(
    "Total projected wins:",
    round(
        season_projections_2026["projected_wins"].sum(),
        3
    )
)

print(
    "Total expected division winners:",
    round(
        season_projections_2026[
            "division_win_probability"
        ].sum(),
        3
    )
)

print(
    "Total expected playoff teams:",
    round(
        season_projections_2026[
            "playoff_probability"
        ].sum(),
        3
    )
)

season_projections_2026

Teams: 32
Total projected wins: 272.0
Total expected division winners: 8.0
Total expected playoff teams: 14.0


,projection_rank,team,projected_wins,projected_losses,median_wins,win_p10,win_p90,strength_of_schedule,strength_of_schedule_rank,division_win_probability,playoff_probability
0,1,DET,10.7638,6.2362,11.0,8.0,13.0,-0.966600,30,0.5099,0.7904
1,2,BUF,10.5924,6.4076,11.0,8.0,13.0,0.281188,12,0.6325,0.8157
2,3,SEA,10.5548,6.4452,11.0,8.0,13.0,0.054373,18,0.4122,0.7605
3,4,LA,10.3205,6.6795,10.0,8.0,13.0,0.632655,6,0.3545,0.7175
4,5,BAL,10.2017,6.7983,10.0,8.0,13.0,-0.697791,28,0.5188,0.7452
5,6,DEN,10.1649,6.8351,10.0,8.0,13.0,-0.180745,20,0.5334,0.7426
6,7,HOU,9.9525,7.0475,10.0,7.0,12.0,-0.317910,24,0.4438,0.7018
7,8,PHI,9.8576,7.1424,10.0,7.0,12.0,-0.565102,26,0.5648,0.6883
8,9,SF,9.5526,7.4474,10.0,7.0,12.0,0.252536,14,0.2155,0.5793
9,10,GB,9.4979,7.5021,10.0,7.0,12.0,0.447042,7,0.2462,0.5642
